# 📈 Material para Capítulo de Resultados — CASSERISISSIMA 2.0

**Trabajo Especial de Grado**  
Universidad de Oriente — Núcleo de Monagas  
Autores: Br. Jorfran Gil · Br. Yefferson Hernández  

---

Este notebook genera tablas, gráficas y diagramas de calidad académica para incluir en el capítulo de **Resultados** de la tesis. Todas las figuras se exportan como PNG de alta resolución.

> **Nota**: Para exportar las figuras, ejecutar todas las celdas. Las imágenes se guardan en `docs/images/`.

In [ ]:
import sys, os

PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), '..'))
SRC_DIR = os.path.join(PROJECT_ROOT, 'src')
IMAGES_DIR = os.path.join(PROJECT_ROOT, 'docs', 'images')
sys.path.insert(0, SRC_DIR)
os.makedirs(IMAGES_DIR, exist_ok=True)

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import warnings
warnings.filterwarnings('ignore')

# Estilo académico para las figuras
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams.update({
    'figure.figsize': (12, 5),
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 11,
})

def save_fig(fig, name):
    """Guarda una figura en docs/images/ con alta resolución."""
    path = os.path.join(IMAGES_DIR, f'{name}.png')
    fig.savefig(path, dpi=300, bbox_inches='tight', facecolor='white')
    print(f'  ✓ Guardada: {path}')

print(f'Imágenes se guardarán en: {IMAGES_DIR}')

In [ ]:
from db.database import SessionLocal, init_db
from db.models import Product, SaleTransaction
from core.ml.model_trainer import train_product_model

init_db()
db = SessionLocal()

products = db.query(Product).filter(Product.is_active == True).all()
print(f'Productos activos: {len(products)}')

## 1. Descripción de los Escenarios Experimentales

El sistema fue evaluado bajo tres escenarios que representan diferentes condiciones operativas de la pastelería.

In [ ]:
from db.seed import SCENARIO_META

scenarios_table = pd.DataFrame([
    {
        'Escenario': f'Escenario {sid}',
        'Nombre': meta['name'],
        'Descripción': meta['description'],
        'Período': f"{meta.get('start', 'N/A')} — {meta.get('end', 'N/A')}",
    }
    for sid, meta in SCENARIO_META.items()
])

# Contar registros por escenario
for sid in SCENARIO_META:
    count = db.query(SaleTransaction).filter(SaleTransaction.scenario_id == sid).count()
    print(f'Escenario {sid}: {count:,} registros de venta')

scenarios_table

## 2. Resultados del Entrenamiento por Escenario

Se entrena el modelo para cada producto en el **Escenario 2 (Óptimo)** y se reportan las métricas de evaluación.

In [ ]:
# Entrenar todos los modelos del Escenario 2 y recopilar métricas
SCENARIO_ID = 2
results = []

for idx, prod in enumerate(products, 1):
    sales_rows = (
        db.query(SaleTransaction.sale_date, SaleTransaction.quantity_sold)
        .filter(
            SaleTransaction.scenario_id == SCENARIO_ID,
            SaleTransaction.product_id == prod.id,
        )
        .order_by(SaleTransaction.sale_date)
        .all()
    )
    
    if len(sales_rows) < 7:
        continue
    
    sales_df = pd.DataFrame(
        [(r.sale_date.isoformat(), float(r.quantity_sold)) for r in sales_rows],
        columns=['sale_date', 'quantity_sold']
    )
    
    try:
        result = train_product_model(
            sales_df=sales_df,
            product_id=prod.id,
            sku=prod.sku,
            shelf_life_days=prod.shelf_life_days,
            n_cv_splits=3,
        )
        
        model_type = result['version_tag'].split('_')[0]
        results.append({
            'Producto': prod.name,
            'SKU': prod.sku,
            'Categoría': prod.category,
            'Modelo Ganador': 'LightGBM' if model_type == 'lgbm' else 'Random Forest',
            'MAPE': result['mape_val'],
            'RMSE': result['rmse_val'],
            'MAE': result['mae_val'],
            'Filas': result['training_rows'],
        })
        print(f'  [{idx}/{len(products)}] ✓ {prod.name} — MAPE={result["mape_val"]:.4f}')
    except Exception as e:
        print(f'  [{idx}/{len(products)}] ✗ {prod.name} — Error: {e}')

results_df = pd.DataFrame(results)
print(f'\nModelos entrenados exitosamente: {len(results_df)}/{len(products)}')

In [ ]:
# Tabla de resultados formateada
display_df = results_df.copy()
display_df['MAPE'] = display_df['MAPE'].apply(lambda x: f'{x:.4f} ({x*100:.1f}%)')
display_df['RMSE'] = display_df['RMSE'].apply(lambda x: f'{x:.4f}')
display_df['MAE'] = display_df['MAE'].apply(lambda x: f'{x:.4f}')

print('\nTabla 1: Resultados del Entrenamiento — Escenario 2 (Óptimo)')
print('='*80)
display_df

In [ ]:
# Figura 1: Comparación de MAE por producto
fig, ax = plt.subplots(figsize=(12, 6))

sorted_df = results_df.sort_values('MAE', ascending=True)
colors = ['#4ECDC4' if m == 'Random Forest' else '#FF6B6B' for m in sorted_df['Modelo Ganador']]

bars = ax.barh(sorted_df['Producto'], sorted_df['MAE'], color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('MAE (Error Absoluto Medio en unidades de torta)')
ax.set_title('Figura 1: Error Absoluto Medio (MAE) por Producto — Escenario Óptimo', fontweight='bold')

# Leyenda
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor='#4ECDC4', label='Random Forest'),
    Patch(facecolor='#FF6B6B', label='LightGBM'),
]
ax.legend(handles=legend_elements, loc='lower right')

# Valores en las barras
for bar, val in zip(bars, sorted_df['MAE']):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.3f}', va='center', fontsize=9)

plt.tight_layout()
save_fig(fig, 'fig01_mae_por_producto')
plt.show()

In [ ]:
# Figura 2: Comparación de métricas (MAPE, RMSE, MAE) agrupadas
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

metrics = [('MAPE', 'MAPE (Error Porcentual)', '#FF9800'),
           ('RMSE', 'RMSE (Error Cuadrático)', '#2196F3'),
           ('MAE', 'MAE (Error Absoluto)', '#4CAF50')]

for ax, (metric, title, color) in zip(axes, metrics):
    sorted_by = results_df.sort_values(metric, ascending=True)
    ax.barh(sorted_by['SKU'], sorted_by[metric], color=color, alpha=0.8)
    ax.set_xlabel(metric)
    ax.set_title(title, fontweight='bold')

plt.suptitle('Figura 2: Comparación de Métricas de Error por Producto', 
             fontweight='bold', fontsize=14, y=1.02)
plt.tight_layout()
save_fig(fig, 'fig02_metricas_comparadas')
plt.show()

In [ ]:
# Figura 3: Distribución de modelos ganadores (RF vs LightGBM)
winner_counts = results_df['Modelo Ganador'].value_counts()

fig, ax = plt.subplots(figsize=(6, 6))
colors = ['#4ECDC4' if w == 'Random Forest' else '#FF6B6B' for w in winner_counts.index]
wedges, texts, autotexts = ax.pie(
    winner_counts.values, 
    labels=winner_counts.index,
    autopct='%1.1f%%',
    colors=colors,
    startangle=90,
    textprops={'fontsize': 12}
)
for autotext in autotexts:
    autotext.set_fontweight('bold')

ax.set_title('Figura 3: Distribución de Modelos Ganadores\n(seleccionados por menor RMSE)', 
             fontweight='bold')

save_fig(fig, 'fig03_modelos_ganadores')
plt.show()

print(f'\nResumen:')
for model, count in winner_counts.items():
    print(f'  {model}: {count} productos ({count/len(results_df)*100:.1f}%)')

In [ ]:
# Estadísticas resumen para el texto de la tesis
print('='*60)
print('ESTADÍSTICAS PARA EL CAPÍTULO DE RESULTADOS')
print('='*60)
print(f'\nTotal de productos evaluados: {len(results_df)}')
print(f'\nMAPE promedio: {results_df["MAPE"].mean():.4f} ({results_df["MAPE"].mean()*100:.1f}%)')
print(f'MAPE mediana:  {results_df["MAPE"].median():.4f} ({results_df["MAPE"].median()*100:.1f}%)')
print(f'MAPE mínimo:   {results_df["MAPE"].min():.4f} ({results_df["MAPE"].min()*100:.1f}%)')
print(f'MAPE máximo:   {results_df["MAPE"].max():.4f} ({results_df["MAPE"].max()*100:.1f}%)')
print(f'\nRMSE promedio: {results_df["RMSE"].mean():.4f}')
print(f'MAE promedio:  {results_df["MAE"].mean():.4f}')
print(f'MAE mediana:   {results_df["MAE"].median():.4f}')
print(f'\nProductos con MAE < 1.0 (error menor a 1 torta): {(results_df["MAE"] < 1.0).sum()}/{len(results_df)}')
print(f'Productos con MAE < 0.5 (error menor a media torta): {(results_df["MAE"] < 0.5).sum()}/{len(results_df)}')

## 3. Diagramas del Sistema

Los siguientes diagramas se pueden copiar directamente al documento de tesis. Están escritos en Mermaid y pueden renderizarse con herramientas como [mermaid.live](https://mermaid.live).

### Diagrama 1: Arquitectura del Sistema

```mermaid
graph TD
    subgraph Frontend ["Capa de Presentación — Next.js 16"]
        UI["Componentes React + Tailwind CSS"]
        API_Client["Cliente API — Axios"]
    end

    subgraph Backend ["Capa de Lógica — FastAPI"]
        Router["Endpoints REST — /api/v1/"]
        ML_Engine["Motor ML — Random Forest / LightGBM"]
        OR_Engine["Investigación de Operaciones — ROP / Newsvendor"]
        DB_Layer["ORM — SQLAlchemy"]
    end

    subgraph Data ["Capa de Persistencia"]
        SQLite[("SQLite — casserisissima.db")]
    end

    UI <--> API_Client
    API_Client <--> Router
    Router <--> ML_Engine
    Router <--> OR_Engine
    ML_Engine <--> DB_Layer
    OR_Engine <--> DB_Layer
    DB_Layer <--> SQLite
```

### Diagrama 2: Pipeline de Machine Learning

```mermaid
flowchart TD
    A["Datos históricos de ventas"] --> B["Rellenar días sin venta"]
    B --> C["Winsorización adaptativa"]
    C --> D["Feature Engineering"]
    D --> D1["Lags temporales"]
    D --> D2["Estadísticas móviles"]
    D --> D3["Variables de calendario"]
    D --> D4["Encoding cíclico"]
    D --> D5["Features de interacción"]
    
    D1 & D2 & D3 & D4 & D5 --> E["DataFrame de Features"]
    E --> F{"Evaluación de calidad de datos"}
    
    F -->|"Alto: ≥50 filas"| G["RandomizedSearchCV"]
    F -->|"Medio: 21-49 filas"| H["RF conservador"]
    F -->|"Bajo: <21 filas"| I["RF lite / EWM"]
    
    G --> G1["Random Forest"] & G2["LightGBM"]
    G1 & G2 --> J{"Comparar por RMSE"}
    J --> K["Serializar modelo ganador"]
    H --> K
    I --> K
    K --> L["Registro en model_registry"]
```

### Diagrama 3: Flujo de una Predicción

```mermaid
sequenceDiagram
    participant U as Usuario
    participant F as Frontend
    participant API as FastAPI
    participant ML as Motor ML
    participant OR as Inv. Operaciones
    participant DB as SQLite

    U->>F: Solicita pronóstico
    F->>API: GET /api/v1/predictions/{sku}
    API->>DB: Cargar historial de ventas
    DB-->>API: Datos de ventas
    API->>ML: Entrenar / cargar modelo
    ML-->>API: Predicción + intervalos
    API->>OR: Calcular Q* y ROP
    OR-->>API: Recomendaciones
    API-->>F: JSON con pronóstico
    F-->>U: Gráficos y recomendaciones
```

### Diagrama 4: Modelo Entidad-Relación

```mermaid
erDiagram
    products ||--o{ sales_transactions : "tiene"
    products ||--o{ demand_forecasts : "recibe"
    products ||--o{ model_registry : "entrena"

    products {
        int id PK
        string sku UK
        string name
        string category
        float selling_price
        float unit_cost
        int shelf_life_days
    }

    sales_transactions {
        int id PK
        int scenario_id
        int product_id FK
        date sale_date
        float quantity_sold
        float revenue
    }

    model_registry {
        int id PK
        int product_id FK
        string version_tag
        float mape_val
        float rmse_val
        float mae_val
    }
```

In [ ]:
# Cerrar sesión
db.close()
print('\n✓ Sesión cerrada.')
print(f'✓ Figuras exportadas en: {IMAGES_DIR}')
print('\nFiguras disponibles para insertar en la tesis:')
for f in sorted(os.listdir(IMAGES_DIR)):
    if f.endswith('.png'):
        print(f'  📊 {f}')

---

*Notebook generado como material para el Capítulo de Resultados — Trabajo Especial de Grado, Universidad de Oriente, Núcleo de Monagas, 2026*